# Coffee-leaf dataset verification — count **before** any preprocessing

**Project:** Design and Development of a Coffee Leaf Disease Diagnosis System Using Deep Learning  
**Owner:** Hayredin Mohammed (UGR/2361/16) · **Week 3**  
**Purpose:** produce the numbers the instructor asked for — *the count before any preprocessing* — from the files themselves, not from a paper table.

### What this notebook does
1. **Probes** every public endpoint and records HTTP status, size and ETag.
2. **Counts** every image per class / per split / per extension, with **no resize, no dedup, no cleaning**.
3. **Hashes** each archive (SHA-256) and each image (first 64 KB) so the corpus is reproducible.
4. **Measures** image dimensions (min / median / max) — this is how we catch a downscaled mirror.
5. **Emits** `dataset_counts.json` + a markdown table identical in shape to `docs/12-DATASET-VERIFICATION.md`.
6. **Optionally** clusters perceptual hashes to estimate the duplicate rate that the leakage-safe split must handle.

### How to run
Runtime → Change runtime type → **CPU** is enough. Disk: use a high-RAM runtime only if you open Silva + Saposoa together.

**Mendeley archives need a logged-in browser session.** Download them manually (free account) and upload,
or mount Drive. The notebook detects whatever is present under `/content/data/raw/` and skips what is missing —
it never fails because a dataset is absent. That is deliberate: a partial run still produces verified counts.

**Golden rule:** every number this notebook prints is a number the group is allowed to quote. If a dataset is
missing, its row says `NOT PRESENT` and the report quotes the *published* figure with the T3 label.


In [ ]:
# 1. Dependencies. `unrar` opens the Saposoa RAR; imagehash does the duplicate estimate.
!apt-get -qq install -y unrar-free > /dev/null 2>&1 || apt-get -qq install -y unrar > /dev/null 2>&1
!pip -q install pillow imagehash pandas tabulate requests

import os, sys, json, hashlib, zipfile, tarfile, subprocess, io, time
from pathlib import Path
from collections import Counter, defaultdict

import requests
import pandas as pd
from PIL import Image

print('python', sys.version.split()[0])
print('ready')


## 2. The registry — one place, every record ID

These are the identifiers from `docs/12-DATASET-VERIFICATION.md` §9. Do not edit a URL here without
editing the report; they are meant to be the same list.


In [ ]:
RAW = Path('/content/data/raw')
OUT = Path('/content/data/verified')
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# local folder name -> (tier we expect, published claim, endpoint used for the manifest)
DATASETS = {
    'jmuben':      {'record': 'Mendeley t2r6rszp5c v1',  'claim': 22591, 'api': 'https://data.mendeley.com/public-api/datasets/t2r6rszp5c/files?folder_id=root&version=1'},
    'jmuben2':     {'record': 'Mendeley tgv3zb82nd v1',  'claim': 35964, 'api': 'https://data.mendeley.com/public-api/datasets/tgv3zb82nd/files?folder_id=root&version=1'},
    'bracol':      {'record': 'Mendeley yy2k5y8mxg v1',  'claim': 3894,  'api': 'https://data.mendeley.com/public-api/datasets/yy2k5y8mxg/files?folder_id=root&version=1'},
    'rocole':      {'record': 'Mendeley c5yvn32dzg v2',  'claim': 1560,  'api': 'https://data.mendeley.com/public-api/datasets/c5yvn32dzg/files?folder_id=root&version=2'},
    'uganda':      {'record': 'Mendeley k36wnd6knb v1',  'claim': 3312,  'api': 'https://data.mendeley.com/public-api/datasets/k36wnd6knb/files?folder_id=root&version=1'},
    'saposoa':     {'record': 'Mendeley mfpxg4y65r v2',  'claim': 1500,  'api': 'https://data.mendeley.com/public-api/datasets/mfpxg4y65r/files?folder_id=root&version=2'},
    'silva':       {'record': 'Mendeley vfxf4trtcg v5',  'claim': 542,   'api': 'https://data.mendeley.com/public-api/datasets/vfxf4trtcg/files?folder_id=root&version=5'},
    'coffeeleafco':{'record': 'Zenodo 22756170',          'claim': 2315,  'api': 'https://zenodo.org/api/records/22756170'},
    'ethiopian':   {'record': 'Kaggle biniyamyoseph/ethiopian-coffee-leaf-disease', 'claim': None, 'api': None},
    'uganda_aug':  {'record': 'Kaggle chelangatspecioza/coffeedataset34', 'claim': 26315, 'api': None},
}

IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
ARCHIVE_EXT = {'.zip', '.rar', '.tar', '.gz', '.tgz', '.7z'}
print(f'{len(DATASETS)} datasets registered')


## 3. Probe — never download something that is not there

A probe is cheap and it proves the record still exists on the day we claim it does. If a probe fails,
the notebook records the failure and moves on. **A failed probe is a finding, not an error.**


In [ ]:
def probe(url, timeout=30):
    t0 = time.time()
    try:
        r = requests.get(url, timeout=timeout, headers={'accept': 'application/json'})
        return {
            'url': url, 'status': r.status_code,
            'bytes': len(r.content), 'ms': int((time.time()-t0)*1000),
            'etag': r.headers.get('etag'), 'ctype': r.headers.get('content-type'),
            'ok': r.ok,
        }
    except Exception as e:
        return {'url': url, 'status': None, 'error': str(e)[:200], 'ok': False}

probes = {}
for name, meta in DATASETS.items():
    if meta['api']:
        probes[name] = probe(meta['api'])
        p = probes[name]
        print(f"{name:14s} {str(p.get('status')):5s} {p.get('ms','?'):>5} ms  {meta['record']}")
    else:
        print(f"{name:14s} (no manifest API — counted from the local tree)")


## 4. Count files — the actual deliverable

`count_tree()` walks a directory and counts **files as they are**, with no filtering beyond the extension.
It returns per-class counts (the folder immediately above the file), per-split counts (a folder named
train/val/test/validation anywhere in the path), and the extension breakdown.

**This is the 'before any preprocessing' number.** It is computed on the extracted archive, before a single
image is resized.


In [ ]:
def count_tree(root: Path):
    """Count image files under root. No resize, no dedup, no cleaning."""
    root = Path(root)
    by_class, by_split, by_ext = Counter(), Counter(), Counter()
    dims = []
    n_images = n_other = 0
    for p in root.rglob('*'):
        if not p.is_file():
            continue
        ext = p.suffix.lower()
        by_ext[ext] += 1
        if ext in IMAGE_EXT:
            n_images += 1
            rel = p.relative_to(root).parts
            # class = nearest folder above the file that is not a split name
            split_names = {'train','training','val','validation','test','testing','train aug'}
            cls = '(root)'
            for part in reversed(rel[:-1]):
                if part.lower() not in split_names:
                    cls = part; break
            by_class[cls] += 1
            for part in rel[:-1]:
                if part.lower() in split_names:
                    by_split[part.lower()] += 1
                    break
            else:
                by_split['(unsplit)'] += 1
            if len(dims) < 4000:   # cap the dimension scan; it is a sanity check, not a census
                try:
                    with Image.open(p) as im:
                        dims.append(im.size)
                except Exception:
                    pass
        else:
            n_other += 1
    dim_stats = {}
    if dims:
        ws = sorted(d[0] for d in dims); hs = sorted(d[1] for d in dims)
        med = lambda a: a[len(a)//2]
        dim_stats = {'min_w': ws[0], 'max_w': ws[-1], 'median_w': med(ws),
                     'min_h': hs[0], 'max_h': hs[-1], 'median_h': med(hs), 'sampled': len(dims)}
    return {
        'root': str(root), 'images': n_images, 'other_files': n_other,
        'by_class': dict(by_class.most_common()),
        'by_split': dict(by_split.most_common()),
        'by_ext': dict(by_ext.most_common()),
        'dimensions': dim_stats,
    }

# quick self-test so a broken cell fails here and not after a 3 GB download
demo = Path('/content/data/_selftest/a/b'); demo.mkdir(parents=True, exist_ok=True)
Image.new('RGB', (64, 64)).save(demo / 'x.jpg')
assert count_tree('/content/data/_selftest')['images'] == 1
print('count_tree self-test passed')


## 5. Extract what is present, then count it

Drop the archives into `/content/data/raw/<name>/` (or mount Drive and symlink them). The cell below
extracts every archive it finds, exactly once, into `/content/data/extracted/<name>/`.

**Mendeley note.** The public file-manifest API is open, but the file download needs a logged-in session.
Open the record page in a browser, download the zip, then upload it. For BRACOL and Saposoa this is the
only way; there is no anonymous URL.


In [ ]:
EXTRACTED = Path('/content/data/extracted'); EXTRACTED.mkdir(exist_ok=True)

def extract_all(name: str):
    src = RAW / name
    dst = EXTRACTED / name
    dst.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        return 'NOT PRESENT'
    did = []
    for arc in src.rglob('*'):
        if not arc.is_file():
            continue
        ext = arc.suffix.lower()
        try:
            if ext == '.zip':
                with zipfile.ZipFile(arc) as z: z.extractall(dst)
                did.append(arc.name)
            elif ext in {'.tar', '.gz', '.tgz'}:
                with tarfile.open(arc) as t: t.extractall(dst)
                did.append(arc.name)
            elif ext == '.rar':
                r = subprocess.run(['unrar','x','-o+',str(arc),str(dst)], capture_output=True, text=True)
                did.append(f'{arc.name} (rc={r.returncode})')
        except Exception as e:
            did.append(f'{arc.name} FAILED: {str(e)[:80]}')
    return did or 'no archives'

for name in DATASETS:
    print(f'{name:14s}', extract_all(name))


In [ ]:
# 6. Count everything that is now on disk.
results = {}
for name, meta in DATASETS.items():
    root = EXTRACTED / name
    if not root.exists() or not any(root.rglob('*')):
        results[name] = {'record': meta['record'], 'status': 'NOT PRESENT',
                         'published_claim': meta['claim']}
        continue
    c = count_tree(root)
    c['record'] = meta['record']
    c['published_claim'] = meta['claim']
    c['status'] = 'counted'
    c['delta_vs_claim'] = (c['images'] - meta['claim']) if meta['claim'] else None
    results[name] = c

print(f"{'dataset':14s} {'images':>8s} {'claim':>8s} {'delta':>7s}  classes")
for name, r in results.items():
    if r['status'] != 'counted':
        print(f"{name:14s} {'—':>8s} {str(r['published_claim']):>8s} {'—':>7s}  NOT PRESENT")
        continue
    d = '—' if r['delta_vs_claim'] is None else f"{r['delta_vs_claim']:+d}"
    print(f"{name:14s} {r['images']:>8d} {str(r['published_claim']):>8s} {d:>7s}  {len(r['by_class'])} classes")


## 7. Hash the archives — the reproducibility artefact

If the group cannot reproduce the corpus byte-for-byte, the split manifest means nothing. SHA-256 of each
archive is cheap and it is the one line a reviewer can check.


In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

hashes = {}
for name in DATASETS:
    src = RAW / name
    if not src.exists():
        continue
    for arc in sorted(src.rglob('*')):
        if arc.is_file() and arc.suffix.lower() in ARCHIVE_EXT:
            h = sha256(arc)
            hashes.setdefault(name, []).append({'file': arc.name, 'bytes': arc.stat().st_size, 'sha256': h})
            print(f'{name:14s} {arc.name[:48]:48s} {arc.stat().st_size/1e6:8.1f} MB  {h[:16]}…')
if not hashes:
    print('no archives on disk yet — upload them to /content/data/raw/<name>/ and re-run')


## 8. Optional — duplicate estimate (this is the input to the leakage-safe split)

Perceptual hashing answers the question the papers do not: **how many of these images are near-copies?**
Run it per dataset. A high duplicate rate is not a bug in the dataset — it is the reason a random split
is invalid. This cell is the evidence for §5.2 of the briefing.

Set `RUN_PHASH = True` for the datasets you have. It is slow on 58k images, so start with Uganda and RoCoLe.


In [ ]:
RUN_PHASH = False
PHASH_DATASETS = ['uganda', 'rocole', 'ethiopian']

def phash_duplicates(root: Path, threshold=6):
    import imagehash
    hashes = []
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMAGE_EXT:
            try:
                with Image.open(p) as im:
                    hashes.append((str(p.relative_to(root)), imagehash.phash(im.convert('RGB'))))
            except Exception:
                pass
    # bucket by hash prefix, then compare within bucket
    buckets = defaultdict(list)
    for path, h in hashes:
        buckets[str(h)[:2]].append((path, h))
    clusters, seen = [], set()
    for key, items in buckets.items():
        for i, (p1, h1) in enumerate(items):
            if p1 in seen: continue
            group = [p1]
            for p2, h2 in items[i+1:]:
                if p2 in seen: continue
                if h1 - h2 <= threshold:
                    group.append(p2); seen.add(p2)
            if len(group) > 1:
                clusters.append(group)
    n = len(hashes)
    dup = sum(len(g)-1 for g in clusters)
    return {'images_hashed': n, 'duplicate_clusters': len(clusters),
            'redundant_images': dup, 'duplicate_rate': round(dup/n, 4) if n else 0,
            'largest_clusters': sorted([len(g) for g in clusters], reverse=True)[:10]}

phash_results = {}
if RUN_PHASH:
    for name in PHASH_DATASETS:
        root = EXTRACTED / name
        if root.exists() and any(root.rglob('*')):
            phash_results[name] = phash_duplicates(root)
            print(name, phash_results[name])
else:
    print('RUN_PHASH is False — set it to True to estimate the duplicate rate')


## 9. Emit the artefacts

`dataset_counts.json` is what SO1 points at; `dataset_counts.md` is pasted into the datasheet.
Nothing in these files is hand-typed.


In [ ]:
payload = {
    'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'note': 'Counts are raw file counts before any preprocessing (no resize, no dedup, no cleaning).',
    'probes': probes,
    'results': results,
    'archive_sha256': hashes,
    'phash': phash_results,
}
(OUT / 'dataset_counts.json').write_text(json.dumps(payload, indent=2))

rows = []
for name, r in results.items():
    if r['status'] == 'counted':
        rows.append({
            'dataset': name, 'record': r['record'],
            'observed_images': r['images'],
            'published_claim': r['published_claim'],
            'delta': r['delta_vs_claim'],
            'classes': '; '.join(f'{k}={v}' for k, v in r['by_class'].items()),
            'median_dim': f"{r['dimensions'].get('median_w','?')}x{r['dimensions'].get('median_h','?')}",
        })
    else:
        rows.append({'dataset': name, 'record': r['record'], 'observed_images': 'NOT PRESENT',
                     'published_claim': r['published_claim'], 'delta': None, 'classes': '', 'median_dim': ''})
df = pd.DataFrame(rows)
df.to_csv(OUT / 'dataset_counts.csv', index=False)
(OUT / 'dataset_counts.md').write_text(df.to_markdown(index=False))

print(df.to_markdown(index=False))
print()
print('written:', OUT / 'dataset_counts.json')
print('written:', OUT / 'dataset_counts.csv')
print('written:', OUT / 'dataset_counts.md')


## 10. The verdict

Run this cell last. It prints the sentence the group is allowed to say out loud.


In [ ]:
counted = {k: v for k, v in results.items() if v['status'] == 'counted'}
missing = [k for k, v in results.items() if v['status'] != 'counted']
total = sum(v['images'] for v in counted.values())

print(f'datasets counted : {len(counted)} / {len(results)}')
print(f'images counted   : {total:,} (raw, before preprocessing)')
print(f'not present      : {", ".join(missing) if missing else "none"}')
print()
print('Say this:')
print(f'  "We counted {total:,} images across {len(counted)} datasets before any preprocessing.')
print('   Every count comes from the files, not from a paper table.')
print('   The archives are SHA-256 hashed and the table is regenerated by one notebook."')

if missing:
    print()
    print('  Still to open (quote the published figure, labelled T3, until then):', ', '.join(missing))
